# 06 · Entrenamiento del modelo v4 (resolución 1280px)

**Objetivo:** entrenar una cuarta versión del detector aumentando la resolución de entrada de 640 a 1280 píxeles, con el fin de mejorar específicamente la detección del balón.

**Contexto:** el análisis de los modelos anteriores reveló que el balón es la clase más difícil por su tamaño minúsculo (pocos píxeles). A resolución 640, el balón casi desaparece para las capas profundas de la red. Al duplicar la resolución a 1280, el balón ocupa cuatro veces más píxeles, lo que debería facilitar su detección. El data augmentation del v3 no mejoró el balón porque su problema es de tamaño, no de color, por lo que esta es la técnica adecuada para esta clase.

**Entrada:** dataset versión 2 de Roboflow (743 imágenes sin augmentation, para aislar el efecto de la resolución).

**Salida:** modelo `best_v4.pt` y métricas.

## 1. Descarga del dataset

Se utiliza la versión 2 del dataset (743 imágenes sin augmentation). Se elige esta versión y no la aumentada para aislar el efecto de la resolución: la única variable que cambia respecto al modelo v2 es el tamaño de entrada (1280 frente a 640).

In [1]:
from roboflow import Roboflow
from getpass import getpass
from pathlib import Path
import shutil

REPO_ROOT = Path("../..").resolve()
DATASET_DIR = REPO_ROOT / "data" / "dataset"

WORKSPACE = "hugos-workspace-0dwnk"
PROJECT = "football-tactical-analysis"

api_key = getpass("Introduce tu API key de Roboflow: ")

In [2]:
# Solo descarga si no existe ya la versión 2
if not (DATASET_DIR / "data.yaml").exists():
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(2)
    dataset = version.download("yolov8", location=str(DATASET_DIR))
    print(f"Dataset v2 descargado en: {DATASET_DIR}")
else:
    print(f"Dataset ya presente en: {DATASET_DIR}")
    print("Si quieres re-descargarlo, borra primero la carpeta data/dataset/")

Dataset ya presente en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset
Si quieres re-descargarlo, borra primero la carpeta data/dataset/


## 2. Verificación del dataset y la GPU

In [3]:
import torch
from ultralytics import YOLO

DATA_YAML = DATASET_DIR / "data.yaml"

train_imgs = list((DATASET_DIR / "train" / "images").glob("*.jpg"))
valid_imgs = list((DATASET_DIR / "valid" / "images").glob("*.jpg"))
test_imgs = list((DATASET_DIR / "test" / "images").glob("*.jpg")) if (DATASET_DIR / "test").exists() else []

print(f"Train: {len(train_imgs)} imágenes")
print(f"Valid: {len(valid_imgs)} imágenes")
print(f"Test:  {len(test_imgs)} imágenes")
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

Train: 520 imágenes
Valid: 149 imágenes
Test:  74 imágenes

GPU: NVIDIA GeForce RTX 2060
VRAM total: 6.00 GB


In [4]:
test_line = f"test: {DATASET_DIR}/test/images\n" if test_imgs else ""

yaml_content = f"""train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/valid/images
{test_line}nc: 3
names: ['ball', 'player', 'referee']
"""

with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("data.yaml actualizado")

data.yaml actualizado


## 3. Entrenamiento a 1280px

Se mantienen los mismos hiperparámetros que el v2 salvo dos cambios obligados por la mayor resolución:

- **imgsz=1280:** el cambio que queremos evaluar, para que el balón ocupe más píxeles.
- **batch=2:** se reduce drásticamente de 16 a 2 porque a 1280px cada imagen consume mucha más VRAM. Con los 6 GB de la RTX 2060, un batch mayor provocaría un error de memoria (Out Of Memory).
- **amp=True:** precisión mixta automática, reduce el consumo de memoria sin afectar a la calidad.

El entrenamiento será notablemente más lento que a 640px (aproximadamente 3-4 veces).

In [5]:
model = YOLO("yolov8s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=1280,
    batch=2,
    device=0,
    project=str(REPO_ROOT / "runs" / "detect" / "models"),
    name="detector_v4_1280",
    patience=15,
    save=True,
    plots=True,
    workers=0,
    cache=False,
    amp=True
)

print(f"\nEntrenamiento completado")
print(f"Resultados guardados en: {results.save_dir}")

New https://pypi.org/project/ultralytics/8.4.61 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momen

## 4. Métricas del modelo v4

In [6]:
metrics = model.val(workers=0)

print("\nMétricas finales del modelo v4 (1280px):\n")
print(f"{'Clase':<15} {'Precisión':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("-" * 60)

for i, cls in model.names.items():
    p = metrics.box.p[i]
    r = metrics.box.r[i]
    map50 = metrics.box.ap50[i]
    map5095 = metrics.box.ap[i]
    print(f"{cls:<15} {p:>10.3f} {r:>10.3f} {map50:>10.3f} {map5095:>10.3f}")

print("-" * 60)
print(f"{'Promedio':<15} {metrics.box.mp:>10.3f} {metrics.box.mr:>10.3f} {metrics.box.map50:>10.3f} {metrics.box.map:>10.3f}")

Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 53.310.8 MB/s, size: 58.8 KB)
val: Scanning C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\valid\labels.cache... 149 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 2.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.0it/s 9.9s.9ss
                   all        149       3439      0.912      0.807      0.854      0.511
                  ball        125        125      0.908       0.64      0.709      0.251
                player        149       3012      0.955      0.974      0.979      0.719
               referee        143        302      0.873      0.808      0.874      0.565
Speed: 2.1ms preprocess, 29.4ms inference, 0.0ms loss, 2.3ms postprocess per image
Res

## 5. Comparación del balón entre todos los modelos

El objetivo de este modelo era mejorar el balón. Se compara su mAP50 con el de los modelos anteriores.

In [7]:
print("Evolución del mAP50 del balón:\n")
print(f"{'Modelo':<25} {'mAP50 balón':>12}")
print("-" * 40)
print(f"{'v1 (152, 640px)':<25} {0.551:>12.3f}")
print(f"{'v2 (743, 640px)':<25} {0.618:>12.3f}")
print(f"{'v3 (743+aug, 640px)':<25} {0.597:>12.3f}")
print(f"{'v4 (743, 1280px)':<25} {metrics.box.ap50[0]:>12.3f}")

Evolución del mAP50 del balón:

Modelo                     mAP50 balón
----------------------------------------
v1 (152, 640px)                  0.551
v2 (743, 640px)                  0.618
v3 (743+aug, 640px)              0.597
v4 (743, 1280px)                 0.709


In [8]:
source = Path(results.save_dir) / "weights" / "best.pt"
destination = REPO_ROOT / "models" / "best_v4.pt"

shutil.copy2(source, destination)

print(f"Modelo v4 guardado en: {destination}")
print(f"Tamaño: {destination.stat().st_size / (1024*1024):.2f} MB")

Modelo v4 guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\models\best_v4.pt
Tamaño: 21.63 MB


## 6. Conclusiones

Tras este notebook:

- El modelo v4 a resolución 1280px ha logrado la mayor mejora del balón de todo el proyecto: su mAP50 sube a 0.709 (frente a 0.618 del v2), con el recall pasando de 0.576 a 0.640. Esto confirma que el problema del balón era de tamaño (pocos píxeles a 640px), y que aumentar la resolución es la técnica adecuada para resolverlo.
- El v4 es además el mejor modelo en todas las clases simultáneamente: balón 0.709, jugador 0.979 y árbitro 0.874, con un mAP50 promedio de 0.854, el más alto de los cuatro modelos.
- El coste es una inferencia aproximadamente 2-2.5 veces más lenta (de ~12ms a ~24-29ms por imagen), debido al mayor tamaño de entrada. Aun así, la velocidad sigue siendo suficiente para procesar vídeo.
- El modelo queda guardado como `models/best_v4.pt`. La decisión final sobre qué modelo usar en el sistema (priorizando precisión o velocidad) se toma en el notebook de comparativa, que evalúa los cuatro modelos sobre el conjunto de test común.